In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate flash-linear-attention

import os, glob; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch, time, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

D = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]) + '/'
OUT = '/kaggle/working/results/'; os.makedirs(OUT, exist_ok=True)
tr = pd.read_csv(D + 'train.csv'); va = pd.read_csv(D + 'val.csv'); print(D, tr.shape, va.shape)

NAME = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
backbone = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).model.cuda()

TARGET_MIX = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj']
backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
backbone.enable_input_require_grads()
pm = get_peft_model(backbone, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=TARGET_MIX))
pm.print_trainable_parameters()
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import random, math, json, re
import torch.nn as nn
from torch.nn import functional as F
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment

CLASSES = sorted(tr.model.unique()); cid = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAP = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
             'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
             'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
             'gpt-6-astra','kimi-k3','qwen3.8-max']
assert CLASSES == LABEL_MAP, '순서 불일치!'; print('label_map 순서 일치 ✅')

def company(m):
    for k, v in [('claude','anthropic'),('gemini','google'),('gpt','openai'),('qwen','qwen'),('glm','zai'),('kimi','moonshot')]:
        if m.startswith(k): return v
COMP = np.array([company(c) for c in CLASSES]); COMPS = sorted(set(COMP))
M2C = torch.tensor([COMPS.index(c) for c in COMP], device='cuda')
ytr = tr.model.map(cid).values; yva = va.model.map(cid).values

def hungarian(P, pids):
    pred = P.argmax(1).copy()
    for p in np.unique(pids):
        idx = np.where(pids == p)[0]; r, c = linear_sum_assignment(-np.log(P[idx] + 1e-9)); pred[idx[r]] = c
    return pred

def evaluate(P, y, pids):
    pred = P.argmax(1); cy = np.array([COMPS.index(COMP[i]) for i in y])
    cP = np.stack([P[:, COMP == c].sum(1) for c in COMPS], 1)
    return dict(model_BA=float(balanced_accuracy_score(y, pred)), accuracy=float(accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average='macro')),
                company_BA=float(balanced_accuracy_score(cy, cP.argmax(1))),
                company_BA_argmax=float(balanced_accuracy_score(cy, [COMPS.index(COMP[i]) for i in pred])),
                model_BA_hungarian=float(balanced_accuracy_score(y, hungarian(P, pids))))

def save_run(run_id, P, df, metrics, extra):
    out = pd.DataFrame(P, columns=CLASSES); out.insert(0, 'id', df.id.values)
    fname = f'val_{run_id}.csv' if 'seed' in run_id else f'val_{run_id}_seed42.csv'
    out.to_csv(OUT + fname, index=False)
    json.dump({**metrics, **extra}, open(f'{OUT}{run_id}_metrics.json', 'w'), indent=2, default=float)
    print(run_id, {k: round(v, 4) if isinstance(v, float) else v for k, v in {**metrics, **extra}.items() if k != 'history'}, flush=True)

class FlatHead(nn.Module):
    def __init__(s, d): super().__init__(); s.fc = nn.Linear(d, 18)
    def forward(s, h): return F.log_softmax(s.fc(h), -1), None

class HierHead(nn.Module):
    def __init__(s, d): super().__init__(); s.comp = nn.Linear(d, len(COMPS)); s.model = nn.Linear(d, 18)
    def forward(s, h):
        lc = F.log_softmax(s.comp(h), -1); z = s.model(h); lm = torch.empty_like(z)
        for c in range(len(COMPS)):
            idx = (M2C == c).nonzero(as_tuple=True)[0]
            lm[:, idx] = F.log_softmax(z[:, idx], -1) + lc[:, c:c+1]
        return lm, lc

def smooth_nll(logp, y, ls):
    return ((1 - ls) * -logp.gather(1, y[:, None]).squeeze(1) + ls * -logp.mean(1)).mean()

def crop(t, rng):
    s = re.split(r'(?<=[.!?])\s+', t.strip())
    if len(s) <= 2: return t
    k = rng.randint(math.ceil(0.5 * len(s)), len(s) + 1); st = rng.randint(0, len(s) - k + 1)
    return ' '.join(s[st:st + k])

def encode(texts):
    return tok(texts, padding=True, truncation=True, max_length=384, pad_to_multiple_of=64, return_tensors='pt').to('cuda')

def pool(enc):
    h = pm(**enc).last_hidden_state.float(); m = enc.attention_mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)

def feat(H, center): return torch.cat([H, H - H.mean(0, keepdim=True)], -1) if center else H

@torch.no_grad()
def predict(df, head, center):
    pm.eval(); head.eval(); out = {}
    for _, g in df.groupby('prompt_id'):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(list(g.text)))
        P = head(feat(H, center))[0].exp().cpu().numpy()
        for i, p in zip(g.index, P): out[i] = p
    return np.stack([out[i] for i in df.index])

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_run(run_id, center=True, epochs=3, lr_lora=2e-4, lr_head=1e-3, seed=42,
              ls=0.0, lora_drop=0.05, wd=0.01, crop_aug=False, hier=False, lam=0.5, rdrop=0.0):
    set_seed(seed); torch.cuda.reset_peak_memory_stats()
    for mod in pm.modules():
        if hasattr(mod, 'reset_lora_parameters'): mod.reset_lora_parameters('default', True)
        if hasattr(mod, 'lora_dropout') and 'default' in mod.lora_dropout and isinstance(mod.lora_dropout['default'], nn.Dropout):
            mod.lora_dropout['default'].p = lora_drop
    d = 1024 * (2 if center else 1); head = (HierHead(d) if hier else FlatHead(d)).cuda()
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': params, 'lr': lr_lora}, {'params': head.parameters(), 'lr': lr_head}], weight_decay=wd)
    pids = tr.prompt_id.unique(); total = epochs * len(pids); warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
    scaler = torch.amp.GradScaler(); t0 = time.time(); hist = []

    def step_loss(texts, y):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(texts))
        lm, lc = head(feat(H, center))
        return lm, smooth_nll(lm, y, ls) + (lam * smooth_nll(lc, M2C[y], ls) if hier else 0)

    for ep in range(epochs):
        pm.train(); head.train(); rng = np.random.RandomState(seed + ep); losses = []
        for pid in rng.permutation(pids):
            g = tr[tr.prompt_id == pid]; y = torch.tensor(g.model.map(cid).values, device='cuda')
            texts = [crop(t, rng) for t in g.text] if crop_aug else list(g.text)
            lm1, loss = step_loss(texts, y)
            if rdrop > 0:
                lm2, loss2 = step_loss(texts, y)
                kl = 0.5 * (F.kl_div(lm1, lm2, log_target=True, reduction='batchmean') +
                            F.kl_div(lm2, lm1, log_target=True, reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + rdrop * kl
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        Pva = predict(va, head, center); m = evaluate(Pva, yva, va.prompt_id.values)
        hist.append({'epoch': ep + 1, 'loss': float(np.mean(losses)), **m})
        print(f'[{run_id}] epoch {ep + 1} | loss {np.mean(losses):.3f} | val BA {m["model_BA"]:.4f} | +hung {m["model_BA_hungarian"]:.4f} | {time.time() - t0:.0f}s', flush=True)
    pm.save_pretrained(f'{OUT}{run_id}_lora_ep{epochs}'); torch.save(head.state_dict(), f'{OUT}{run_id}_head_ep{epochs}.pt')
    Ptr = predict(tr, head, center)
    save_run(run_id, Pva, va, m, dict(train_BA=float(balanced_accuracy_score(ytr, Ptr.argmax(1))), sec=round(time.time() - t0), seed=seed,
             peak_GB=round(torch.cuda.max_memory_allocated() / 1e9, 1), center=center, epochs=epochs, lr_lora=lr_lora, lr_head=lr_head,
             ls=ls, lora_drop=lora_drop, wd=wd, crop_aug=crop_aug, hier=hier, lam=lam, rdrop=rdrop, target=TARGET_MIX, r=16, history=hist))
    return head

# ===== V3 seed 43, 44 =====
for s in [43, 44]:
    train_run(f'V3_rdrop_seed{s}', seed=s, rdrop=1.0)
print('V3 seed 43·44 완료 ✅')

In [ ]:
import random, math, json, re
import torch.nn as nn
from torch.nn import functional as F
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment

CLASSES = sorted(tr.model.unique()); cid = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAP = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
             'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
             'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
             'gpt-6-astra','kimi-k3','qwen3.8-max']
assert CLASSES == LABEL_MAP, '순서 불일치!'; print('label_map 순서 일치 ✅')

def company(m):
    for k, v in [('claude','anthropic'),('gemini','google'),('gpt','openai'),('qwen','qwen'),('glm','zai'),('kimi','moonshot')]:
        if m.startswith(k): return v
COMP = np.array([company(c) for c in CLASSES]); COMPS = sorted(set(COMP))
M2C = torch.tensor([COMPS.index(c) for c in COMP], device='cuda')
ytr = tr.model.map(cid).values; yva = va.model.map(cid).values

def hungarian(P, pids):
    pred = P.argmax(1).copy()
    for p in np.unique(pids):
        idx = np.where(pids == p)[0]; r, c = linear_sum_assignment(-np.log(P[idx] + 1e-9)); pred[idx[r]] = c
    return pred

def evaluate(P, y, pids):
    pred = P.argmax(1); cy = np.array([COMPS.index(COMP[i]) for i in y])
    cP = np.stack([P[:, COMP == c].sum(1) for c in COMPS], 1)
    return dict(model_BA=float(balanced_accuracy_score(y, pred)), accuracy=float(accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average='macro')),
                company_BA=float(balanced_accuracy_score(cy, cP.argmax(1))),
                company_BA_argmax=float(balanced_accuracy_score(cy, [COMPS.index(COMP[i]) for i in pred])),
                model_BA_hungarian=float(balanced_accuracy_score(y, hungarian(P, pids))))

def save_run(run_id, P, df, metrics, extra):
    out = pd.DataFrame(P, columns=CLASSES); out.insert(0, 'id', df.id.values)
    fname = f'val_{run_id}.csv' if 'seed' in run_id else f'val_{run_id}_seed42.csv'
    out.to_csv(OUT + fname, index=False)
    json.dump({**metrics, **extra}, open(f'{OUT}{run_id}_metrics.json', 'w'), indent=2, default=float)
    print(run_id, {k: round(v, 4) if isinstance(v, float) else v for k, v in {**metrics, **extra}.items() if k != 'history'}, flush=True)

class FlatHead(nn.Module):
    def __init__(s, d): super().__init__(); s.fc = nn.Linear(d, 18)
    def forward(s, h): return F.log_softmax(s.fc(h), -1), None

class HierHead(nn.Module):
    def __init__(s, d): super().__init__(); s.comp = nn.Linear(d, len(COMPS)); s.model = nn.Linear(d, 18)
    def forward(s, h):
        lc = F.log_softmax(s.comp(h), -1); z = s.model(h); lm = torch.empty_like(z)
        for c in range(len(COMPS)):
            idx = (M2C == c).nonzero(as_tuple=True)[0]
            lm[:, idx] = F.log_softmax(z[:, idx], -1) + lc[:, c:c+1]
        return lm, lc

def smooth_nll(logp, y, ls):
    return ((1 - ls) * -logp.gather(1, y[:, None]).squeeze(1) + ls * -logp.mean(1)).mean()

def crop(t, rng):
    s = re.split(r'(?<=[.!?])\s+', t.strip())
    if len(s) <= 2: return t
    k = rng.randint(math.ceil(0.5 * len(s)), len(s) + 1); st = rng.randint(0, len(s) - k + 1)
    return ' '.join(s[st:st + k])

def encode(texts):
    return tok(texts, padding=True, truncation=True, max_length=384, pad_to_multiple_of=64, return_tensors='pt').to('cuda')

def pool(enc):
    h = pm(**enc).last_hidden_state.float(); m = enc.attention_mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)

def feat(H, center): return torch.cat([H, H - H.mean(0, keepdim=True)], -1) if center else H

@torch.no_grad()
def predict(df, head, center):
    pm.eval(); head.eval(); out = {}
    for _, g in df.groupby('prompt_id'):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(list(g.text)))
        P = head(feat(H, center))[0].exp().cpu().numpy()
        for i, p in zip(g.index, P): out[i] = p
    return np.stack([out[i] for i in df.index])

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_run(run_id, center=True, epochs=3, lr_lora=2e-4, lr_head=1e-3, seed=42,
              ls=0.0, lora_drop=0.05, wd=0.01, crop_aug=False, hier=False, lam=0.5, rdrop=0.0):
    set_seed(seed); torch.cuda.reset_peak_memory_stats()
    for mod in pm.modules():
        if hasattr(mod, 'reset_lora_parameters'): mod.reset_lora_parameters('default', True)
        if hasattr(mod, 'lora_dropout') and 'default' in mod.lora_dropout and isinstance(mod.lora_dropout['default'], nn.Dropout):
            mod.lora_dropout['default'].p = lora_drop
    d = 1024 * (2 if center else 1); head = (HierHead(d) if hier else FlatHead(d)).cuda()
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': params, 'lr': lr_lora}, {'params': head.parameters(), 'lr': lr_head}], weight_decay=wd)
    pids = tr.prompt_id.unique(); total = epochs * len(pids); warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
    scaler = torch.amp.GradScaler(); t0 = time.time(); hist = []

    def step_loss(texts, y):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(texts))
        lm, lc = head(feat(H, center))
        return lm, smooth_nll(lm, y, ls) + (lam * smooth_nll(lc, M2C[y], ls) if hier else 0)

    for ep in range(epochs):
        pm.train(); head.train(); rng = np.random.RandomState(seed + ep); losses = []
        for pid in rng.permutation(pids):
            g = tr[tr.prompt_id == pid]; y = torch.tensor(g.model.map(cid).values, device='cuda')
            texts = [crop(t, rng) for t in g.text] if crop_aug else list(g.text)
            lm1, loss = step_loss(texts, y)
            if rdrop > 0:
                lm2, loss2 = step_loss(texts, y)
                kl = 0.5 * (F.kl_div(lm1, lm2, log_target=True, reduction='batchmean') +
                            F.kl_div(lm2, lm1, log_target=True, reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + rdrop * kl
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        Pva = predict(va, head, center); m = evaluate(Pva, yva, va.prompt_id.values)
        hist.append({'epoch': ep + 1, 'loss': float(np.mean(losses)), **m})
        print(f'[{run_id}] epoch {ep + 1} | loss {np.mean(losses):.3f} | val BA {m["model_BA"]:.4f} | +hung {m["model_BA_hungarian"]:.4f} | {time.time() - t0:.0f}s', flush=True)
    pm.save_pretrained(f'{OUT}{run_id}_lora_ep{epochs}'); torch.save(head.state_dict(), f'{OUT}{run_id}_head_ep{epochs}.pt')
    Ptr = predict(tr, head, center)
    save_run(run_id, Pva, va, m, dict(train_BA=float(balanced_accuracy_score(ytr, Ptr.argmax(1))), sec=round(time.time() - t0), seed=seed,
             peak_GB=round(torch.cuda.max_memory_allocated() / 1e9, 1), center=center, epochs=epochs, lr_lora=lr_lora, lr_head=lr_head,
             ls=ls, lora_drop=lora_drop, wd=wd, crop_aug=crop_aug, hier=hier, lam=lam, rdrop=rdrop, target=TARGET_MIX, r=16, history=hist))
    return head

# ===== Q4 (계층형 헤드 + R-Drop) seed 43, 44 =====
for s in [43, 44]:
    train_run(f'Q4_hier_seed{s}', seed=s, rdrop=1.0, hier=True)
print('Q4 seed 43·44 완료 ✅')

In [1]:
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate flash-linear-attention
import os, glob, time
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in sorted(glob.glob('/kaggle/input/**/*', recursive=True)):
    if f.endswith(('.csv', '.py', '.pt', 'adapter_config.json')): print(f)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
/kaggle/input/datasets/yangseorin/c-assets/Q2_lora_flat_head_ep3.pt
/kaggle/input/datasets/yangseorin/c-assets/Q2_lora_flat_lora_ep3-20261118T025152Z-1-001/Q2_lora_flat_lora_ep3/adapter_config.json
/kaggle/input/datasets/yangseorin/c-assets/Q3_lora_centered_head_ep3.pt
/kaggle/input/datasets/yangseorin/c-assets/Q3_lora_centered_lora_ep3-20261118T025200Z-1-001/Q3_lora_centered_lora_ep3/adapter_config.json
/kaggle/input/datasets/yangseorin/c-assets/Q4_hier_head_ep3.pt
/kaggle/input/datasets/yangseorin/c-assets/Q4_hier_lora_ep3-20261118T025236Z-1-001/Q4_hier_lora_ep3/adapter_config.json
/kaggle/input/datasets/yangseorin/c-assets/V3_rdrop_head_ep3.pt
/kaggle/input/datasets/yangseorin/c-assets/V3_rdrop_lora_ep3-20261118T025301Z-1-001/V3_rdrop_lora_ep3/adapter_config.json
/kaggle/input/datasets/yangseorin/c-assets/val_Q2_lora_flat_seed42.csv
/kaggle/input/datasets/yangseorin/c-assets/val_Q3_lora_centered_seed42.csv
/kaggle/inpu

In [2]:
# ===== 셀 B: val 앙상블 + XGBoost 결합 =====
import glob, os, re, importlib.util, numpy as np, pandas as pd, xgboost as xgb
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
IN = '/kaggle/input'

def find(pat):                      # 여러 입력에 같은 파일이 있으면 하나만
    fs = {}
    for f in sorted(glob.glob(f'{IN}/**/{pat}', recursive=True)): fs.setdefault(os.path.basename(f), f)
    return list(fs.values())
def load_py(name, path):
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
E = load_py('team_eval', find('evaluate.py')[0])                 # 공식 평가 코드
SM = load_py('stylometry', find('stylometry.py')[0])             # B 문체 특징
CLS = E.CLASSES; cid = {c: i for i, c in enumerate(CLS)}
D = os.path.dirname(find('val.csv')[0]) + '/'
tr = pd.read_csv(D + 'train.csv'); va = pd.read_csv(D + 'val.csv').sort_values('id').reset_index(drop=True)
ytr, yva = tr.model.map(cid).values, va.model.map(cid).values
assert len(tr) == 1889 and len(va) == 396

def readP(f, ids):
    df = pd.read_csv(f); assert list(df.columns) == ['id'] + CLS, f
    P = df.set_index('id').loc[ids, CLS].to_numpy(float); return P / P.sum(1, keepdims=True)
def group(pat, n=None):
    fs = find(pat); print(f'{pat:22s} → {[os.path.basename(f) for f in fs]}')
    assert fs and (n is None or len(fs) == n), f'{pat}: 파일 {len(fs)}개 (기대 {n})'
    return [readP(f, va.id) for f in fs]

# ---------- XGBoost (full + 중심화, seed 42/43/44 평균) — 고정 설정 ----------
FW = "the a an and or but if of to in on at by for with from as that which who this these those it its is are was were be been being have has had do does not no can could would should may might will shall must very more most also just only even such so than then there here what when where why how all any each every some many much both either neither one two first however therefore thus moreover yet still often perhaps rather quite into onto through across between among within without beyond about".split()
st = SM.StylometricTransformer().fit(tr.text)
cvec = CountVectorizer(vocabulary=FW, token_pattern=r"\b\w+\b")
tf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), lowercase=False, min_df=2, max_features=30000, sublinear_tf=True).fit(tr.text)
svd = TruncatedSVD(100, random_state=42).fit(tf.transform(tr.text))
def xfeat(df):
    T = df.text
    A = pd.DataFrame(st.transform(T), columns=st.get_feature_names_out())
    n = np.array([max(len(re.findall(r'\b\w+\b', t)), 1) for t in T])[:, None]
    F_ = pd.DataFrame(cvec.transform(T).toarray() / n, columns=['fw_' + w for w in FW])
    S = pd.DataFrame(svd.transform(tf.transform(T)), columns=[f'svd{i}' for i in range(100)])
    X = pd.concat([A, F_, S], axis=1)
    return X - X.groupby(df.prompt_id.values).transform('mean')      # 질문 중심화
Xtr, Xva = xfeat(tr), xfeat(va)
XGB_MODELS = []
for s in [42, 43, 44]:
    m = xgb.XGBClassifier(objective='multi:softprob', n_estimators=3000, max_depth=3, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, min_child_weight=5, reg_lambda=10, early_stopping_rounds=50,
                          eval_metric='mlogloss', n_jobs=4, tree_method='hist', random_state=s)
    XGB_MODELS.append(m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)); print('XGB seed', s, 'trees', m.best_iteration)
P_xgb = np.mean([m.predict_proba(Xva) for m in XGB_MODELS], 0).astype(float); P_xgb /= P_xgb.sum(1, keepdims=True)

# ---------- Qwen 예측 파일 ----------
V3 = group('val_V3_rdrop*.csv', 3); Q4 = group('val_Q4*.csv', 3)
Q2 = group('val_Q2*.csv'); Q3 = group('val_Q3*.csv')
avg = lambda L: np.mean(L, 0)
P_qwen = avg(V3 + Q4)                                                  # ★ 최종 Qwen 부분 (미리 고정)

def hung(P, pids):
    pred = P.argmax(1).copy()
    for p in np.unique(pids):
        i = np.where(pids == p)[0]; r, c = linear_sum_assignment(-np.log(P[i] + 1e-9)); pred[i[r]] = c
    return pred
def ba(y, pred): return np.mean([(pred[y == k] == k).mean() for k in np.unique(y)])

# ---------- XGB 가중치: 0.8 / 0.9 / 1.0 중 val model_BA로 선택 (동점이면 큰 값) ----------
W = {w: float(ba(yva, (w * P_qwen + (1 - w) * P_xgb).argmax(1))) for w in [0.8, 0.9, 1.0]}
W_BEST = max(W, key=lambda w: (round(W[w], 6), w)); print('가중치별 val BA:', {k: round(v, 4) for k, v in W.items()}, '→ 선택', W_BEST)
P_final = W_BEST * P_qwen + (1 - W_BEST) * P_xgb

ROWS = {'XGBoost (full+중심화, 3seed)': P_xgb, 'Q2 LoRA': avg(Q2), 'Q3 LoRA+중심화': avg(Q3),
        'V3 R-Drop seed42': V3[0], 'V3 × 3seed': avg(V3), 'Q4 계층형 seed42': Q4[0], 'Q4 × 3seed': avg(Q4),
        'LoRA 3종 평균 (Q3+V3+Q4, seed42)': avg([avg(Q3), V3[0], Q4[0]]), 'V3×3 + Q4×3': P_qwen,
        f'★ FINAL = V3×3+Q4×3 (+XGB, w={W_BEST})': P_final}
def table(rows, y, pids):
    out = []
    for k, P in rows.items():
        s = E.score(y, P)
        out.append({'run': k, 'model_BA': s['model_BA'], '배정_BA': ba(y, hung(P, pids)), 'company_BA': s['company_BA'],
                    'company_argmax': s['company_BA_argmax'], 'sibling': s['sibling_Acc'], 'F1': s['model_F1'],
                    'LogLoss': s['model_LogLoss'], 'ECE': s['model_ECE']})
    return pd.DataFrame(out).set_index('run').round(4)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 45)
VAL_TABLE = table(ROWS, yva, va.prompt_id.values); print(VAL_TABLE.to_string())

# ---------- 질문 단위 paired bootstrap: FINAL vs V3 seed42 ----------
def boot(PA, PB, y, pids, n=1000, seed=0, hun=False):
    rng = np.random.default_rng(seed); u = np.unique(pids)
    pa = hung(PA, pids) if hun else PA.argmax(1); pb = hung(PB, pids) if hun else PB.argmax(1)
    idx = {p: np.where(pids == p)[0] for p in u}; d = []
    for _ in range(n):
        i = np.concatenate([idx[p] for p in rng.choice(u, len(u))]); d.append(ba(y[i], pb[i]) - ba(y[i], pa[i]))
    return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)
for hun in [False, True]:
    m_, lo, hi = boot(V3[0], P_final, yva, va.prompt_id.values, hun=hun)
    print(f"FINAL − V3 seed42 ({'배정' if hun else '순수'}): {m_:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")

XGB seed 42 trees 627
XGB seed 43 trees 621
XGB seed 44 trees 538
val_V3_rdrop*.csv      → ['val_V3_rdrop_seed42.csv', 'val_V3_rdrop_seed43.csv', 'val_V3_rdrop_seed44.csv']
val_Q4*.csv            → ['val_Q4_hier_seed42.csv', 'val_Q4_hier_seed43.csv', 'val_Q4_hier_seed44.csv']
val_Q2*.csv            → ['val_Q2_lora_flat_seed42.csv']
val_Q3*.csv            → ['val_Q3_lora_centered_seed42.csv']
가중치별 val BA: {0.8: 0.5505, 0.9: 0.5455, 1.0: 0.548} → 선택 0.8
                                   model_BA   배정_BA  company_BA  company_argmax  sibling      F1  LogLoss     ECE
run                                                                                                              
XGBoost (full+중심화, 3seed)            0.3813  0.4318      0.6000          0.6114   0.5078  0.3801   1.7618  0.0703
Q2 LoRA                              0.3283  0.4545      0.5542          0.5682   0.4312  0.3209   1.8990  0.1744
Q3 LoRA+중심화                          0.4823  0.6035      0.6818          0.6784   0.5811

In [3]:
# ===== 셀 C: 저장된 모델로 val 재확인 + test 예측 (학습 없음, 채점 없음) =====
import gc, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
NAME = 'Qwen/Qwen3.5-0.8B'; OUTP = '/kaggle/working/preds/'; os.makedirs(OUTP, exist_ok=True)
tok = AutoTokenizer.from_pretrained(NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# ---- test: 정답 열은 읽자마자 삭제 (글만 사용) ----
TP = find('test*.csv'); assert len(TP) == 1, TP
te = pd.read_csv(TP[0])
te = te.drop(columns=[c for c in ['model'] if c in te.columns]).sort_values('id').reset_index(drop=True)
assert len(te) == 414 and te.prompt_id.value_counts().eq(18).all()
te.to_csv(OUTP + 'test_nolabel.csv', index=False)                 # 팀원 배포용 (정답 없음)

# ---- 학습 때와 똑같은 헤드 정의 ----
COMPS = ['anthropic', 'google', 'moonshot', 'openai', 'qwen', 'zai']
M2C = torch.tensor([COMPS.index({'claude': 'anthropic', 'gemini': 'google', 'gpt': 'openai', 'qwen': 'qwen', 'glm': 'zai',
                                 'kimi': 'moonshot'}[re.match(r'[a-z]+', c).group()]) for c in CLS], device='cuda')
class FlatHead(nn.Module):
    def __init__(s, d): super().__init__(); s.fc = nn.Linear(d, 18)
    def forward(s, h): return F.log_softmax(s.fc(h), -1), None
class HierHead(nn.Module):
    def __init__(s, d): super().__init__(); s.comp = nn.Linear(d, len(COMPS)); s.model = nn.Linear(d, 18)
    def forward(s, h):
        lc = F.log_softmax(s.comp(h), -1); z = s.model(h); lm = torch.empty_like(z)
        for c in range(len(COMPS)):
            idx = (M2C == c).nonzero(as_tuple=True)[0]
            lm[:, idx] = F.log_softmax(z[:, idx], -1) + lc[:, c:c+1]
        return lm, lc

def load_head(path):
    sd = torch.load(path, map_location='cuda')
    if set(sd) == {'weight', 'bias'}: sd = {'fc.' + k: v for k, v in sd.items()}   # Q2·Q3 옛 형식
    hier = 'comp.weight' in sd; d = (sd['model.weight'] if hier else sd['fc.weight']).shape[1]
    head = (HierHead(d) if hier else FlatHead(d)).cuda(); head.load_state_dict(sd); head.eval()
    return head, d == 2048, hier                                   # 2048차원이면 중심화 사용

@torch.no_grad()
def qpredict(pm, head, center, df):
    out = {}
    for _, g in df.groupby('prompt_id'):
        enc = tok(list(g.text), padding=True, truncation=True, max_length=384, pad_to_multiple_of=64, return_tensors='pt').to('cuda')
        with torch.autocast('cuda', dtype=torch.float16):
            h = pm(**enc).last_hidden_state.float()
        m = enc.attention_mask.unsqueeze(-1).float(); H = (h * m).sum(1) / m.sum(1)
        X = torch.cat([H, H - H.mean(0, keepdim=True)], -1) if center else H
        for i, p in zip(g.index, head(X)[0].exp().cpu().numpy()): out[i] = p
    return np.stack([out[i] for i in df.index])

def save_pred(P, df, fname):
    o = pd.DataFrame(P, columns=CLS); o.insert(0, 'id', df.id.values); o.to_csv(OUTP + fname, index=False)

CFGS = glob.glob(f'{IN}/**/adapter_config.json', recursive=True)
def adir_of(key):                                                  # zip 폴더 이름에 날짜가 붙어도 찾기
    return sorted({os.path.dirname(c) for c in CFGS if re.search(key + r'(?![\w])', c.replace(os.sep, '/'))})
RUNS = [('Q2', r'Q2_\w*_lora_ep3', 'Q2*_head_ep3.pt', 'val_Q2*.csv'), ('Q3', r'Q3_\w*_lora_ep3', 'Q3*_head_ep3.pt', 'val_Q3*.csv'),
        ('V3_seed42', r'V3_rdrop_lora_ep3', 'V3_rdrop_head_ep3.pt', 'val_V3_rdrop_seed42.csv'),
        ('V3_seed43', r'V3_rdrop_seed43_lora_ep3', 'V3_rdrop_seed43_head_ep3.pt', 'val_V3_rdrop_seed43.csv'),
        ('V3_seed44', r'V3_rdrop_seed44_lora_ep3', 'V3_rdrop_seed44_head_ep3.pt', 'val_V3_rdrop_seed44.csv'),
        ('Q4_seed42', r'Q4_hier_lora_ep3', 'Q4_hier_head_ep3.pt', 'val_Q4_hier_seed42.csv'),
        ('Q4_seed43', r'Q4_hier_seed43_lora_ep3', 'Q4_hier_seed43_head_ep3.pt', 'val_Q4_hier_seed43.csv'),
        ('Q4_seed44', r'Q4_hier_seed44_lora_ep3', 'Q4_hier_seed44_head_ep3.pt', 'val_Q4_hier_seed44.csv')]
for name, akey, hpat, vpat in RUNS:                                # 먼저 파일이 다 있는지 확인
    assert len(adir_of(akey)) == 1 and len(find(hpat)) == 1 and len(find(vpat)) == 1, \
        f'{name}: 파일 확인 필요 {adir_of(akey)} {find(hpat)} {find(vpat)}'
print('모델 파일 8개 확인 ✅')

TEST, CHECK = {}, []
for name, akey, hpat, vpat in RUNS:
    t0 = time.time(); adir = adir_of(akey)[0]
    head, center, hier = load_head(find(hpat)[0])
    base = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).model.cuda()
    pm = PeftModel.from_pretrained(base, adir).eval()
    Pv = qpredict(pm, head, center, va); Pold = readP(find(vpat)[0], va.id)
    same = (Pv.argmax(1) == Pold.argmax(1)).mean()                 # 학습 때 val 예측과 같은지
    CHECK.append(dict(run=name, center=center, hier=hier, val_BA_now=round(ba(yva, Pv.argmax(1)), 4),
                      val_BA_saved=round(ba(yva, Pold.argmax(1)), 4), same_pred=round(same, 3)))
    TEST[name] = qpredict(pm, head, center, te); save_pred(TEST[name], te, f'test_C_{name}.csv')
    print(CHECK[-1], f'{time.time() - t0:.0f}s', '✅' if same > 0.95 else '⚠️ 불일치! 멈추고 알려줘', flush=True)
    del pm, base, head; gc.collect(); torch.cuda.empty_cache()

# ---- XGBoost test 예측 + 최종 앙상블 (조합·가중치는 val에서 이미 고정) ----
P_te_xgb = np.mean([m.predict_proba(xfeat(te)) for m in XGB_MODELS], 0).astype(float); P_te_xgb /= P_te_xgb.sum(1, keepdims=True)
P_te_qwen = np.mean([TEST[k] for k in ['V3_seed42', 'V3_seed43', 'V3_seed44', 'Q4_seed42', 'Q4_seed43', 'Q4_seed44']], 0)
P_te_final = W_BEST * P_te_qwen + (1 - W_BEST) * P_te_xgb
save_pred(P_te_xgb, te, 'test_C_XGB.csv'); save_pred(P_te_final, te, 'test_C_FINAL.csv')
save_pred(P_xgb, va, 'val_C_XGB.csv'); save_pred(P_final, va, 'val_C_FINAL.csv')
print(pd.DataFrame(CHECK).to_string(index=False)); print('저장된 파일:', sorted(os.listdir(OUTP)))

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

모델 파일 8개 확인 ✅


... (23 outputs left out)

{'run': 'Q4_seed43', 'center': True, 'hier': True, 'val_BA_now': np.float64(0.4949), 'val_BA_saved': np.float64(0.4924), 'same_pred': np.float64(0.997)} 72s ✅


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


{'run': 'Q4_seed44', 'center': True, 'hier': True, 'val_BA_now': np.float64(0.5177), 'val_BA_saved': np.float64(0.5177), 'same_pred': np.float64(1.0)} 73s ✅
      run  center  hier  val_BA_now  val_BA_saved  same_pred
       Q2   False False      0.3283        0.3283      1.000
       Q3    True False      0.4823        0.4823      1.000
V3_seed42    True False      0.4924        0.4924      1.000
V3_seed43    True False      0.4798        0.4798      1.000
V3_seed44    True False      0.4823        0.4823      1.000
Q4_seed42    True  True      0.5000        0.5000      1.000
Q4_seed43    True  True      0.4949        0.4924      0.997
Q4_seed44    True  True      0.5177        0.5177      1.000
저장된 파일: ['test_C_FINAL.csv', 'test_C_Q2.csv', 'test_C_Q3.csv', 'test_C_Q4_seed42.csv', 'test_C_Q4_seed43.csv', 'test_C_Q4_seed44.csv', 'test_C_V3_seed42.csv', 'test_C_V3_seed43.csv', 'test_C_V3_seed44.csv', 'test_C_XGB.csv', 'test_nolabel.csv', 'val_C_FINAL.csv', 'val_C_XGB.csv']


In [4]:
!cd /kaggle/working && zip -qr c_preds.zip preds && ls -la c_preds.zip
from IPython.display import FileLink; FileLink('c_preds.zip')

-rw-r--r-- 1 root root 778022 Nov 18 03:30 c_preds.zip


/kaggle/working/c_preds.zip

In [1]:
!pip install -q -U transformers peft accelerate flash-linear-attention
import sys, platform, torch, transformers, peft, accelerate, xgboost, sklearn, numpy, pandas, scipy
print('OS', platform.platform()); print('python', sys.version.split()[0])
for m in [torch, transformers, peft, accelerate, xgboost, sklearn, numpy, pandas, scipy]: print(m.__name__, m.__version__)

OS Linux-6.18.55+-x86_64-with-glibc2.39
python 3.13.16
torch 2.11.0+cu130
transformers 5.23.0
peft 0.21.3
accelerate 1.16.0
xgboost 3.4.1
sklearn 1.6.1
numpy 2.1.3
pandas 2.3.3
scipy 1.16.3
